In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../")

from src.models import maml_with_l2l
from src.models.models import HighlyFlexibleModel

import pandas as pd
from sklearn.model_selection import train_test_split

from src.scoring.metalearning_scoring_fn import compute_metrics


import torch
from torch import Tensor, nn, no_grad, zeros_like
import torch.nn.functional as F
from torch.optim import SGD, Optimizer
from torch.utils.data import DataLoader

import src.data.sun_et_al as helpers
import src.models.protonet as protonet
import src.models.protomaml as protomaml
import wandb

# wandb.finish()
# wandb.init(project="test")


"""Set the random seed for reproducibility."""
import random

import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)  # PyTorch 1.8.0+


def main():
    # Set device.
    device = "cpu"

    # Create a synthetic dataset.
    # We'll generate a binary classification problem in 2D.
    classification_data = pd.read_csv("data/ionosphere.data", header=None)
    classification_data = classification_data.replace("g", 1)
    classification_data = classification_data.replace("b", 0)
    X = classification_data.iloc[:, :-1].values
    y = classification_data.iloc[:, -1].values

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    train_k_shot = 4

    train_dataset = helpers.LabelOnlyDataset(X_train, y_train)
    eval_dataset = helpers.LabelOnlyDataset(X_val, y_val)
    train_sampler = helpers.KShotBatchSampler(
        train_dataset, train_k_shot, include_query=True
    )
    eval_sampler = helpers.KShotBatchSampler(
        eval_dataset,
        train_k_shot,
        include_query=True,
        query_size="rest",
        shuffle=False,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_sampler=train_sampler,
        # batch_size=32,
    )
    eval_loader = DataLoader(
        eval_dataset,
        batch_sampler=eval_sampler,
        # batch_size=32,
    )

    model = protonet.ProtonetTrainer(
        model=nn.Sequential(nn.Linear(X.shape[1], 30), nn.ReLU(), nn.Linear(30, 1)),
        device=device,
        starting_lr=0.01,
        scheduler_step=50,
        scheduler_gamma=0.5,
        train_k_shot=train_k_shot,
    )

    # model = maml_with_l2l.MAML(
    #     HighlyFlexibleModel(X.shape[1]),
    #     train_n_gradient_steps=3,
    #     eval_n_gradient_steps=3,
    #     device=device,
    #     inner_lr_range=[1e-3, 1e-3],
    #     outer_lr_range=[1e-2, 1e-5],
    #     inner_lr_reduction_factor=1,
    #     train_k_shot=train_k_shot,
    #     loss_fn=nn.CrossEntropyLoss(),
    # )

    # model = protomaml.ProtoMAMLTrainer(HighlyFlexibleModel(X.shape[1]),
    #                                    device=device, train_k_shot=train_k_shot, inner_lr=1e-3, num_inner_steps=3, lr_output=1e-2, eval_k_shot=train_k_shot)

    model.fit(
        train_dataloader=train_loader,
        n_epochs=1000,
        n_parallel_tasks=5,
        eval_dataloader=eval_loader,
        val_or_test="val",
        log_gradients=True,
    )

    # test with a normal neural net
    # model = nn.Sequential(
    #     nn.Linear(X_train.shape[1], 4),
    #     nn.ReLU(),
    #     nn.Linear(4, 1),
    # ).to(device)
    # optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)
    # criterion = nn.BCEWithLogitsLoss()

    # # plotting loss and metrics
    # train_loss = []
    # eval_loss = []
    # metrics = {"accuracy": [], "f1": [], "precision": [], "recall": [], "roc_auc": []}

    # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    # for epoch in range(400):
    #     model.train()
    #     for batch in train_loader:
    #         x, y = batch
    #         x, y = x.to(device), y.to(device)
    #         optimizer.zero_grad()
    #         logits = model(x).squeeze()
    #         loss = criterion(logits, y)
    #         loss.backward()

    #         optimizer.step()

    #         train_loss.append(loss.detach().item())

    #     scheduler.step()

    #     if epoch % 10 == 0:
    #         print(f"Epoch {epoch}, Loss: {loss.item()}")

    #         model.eval()
    #         with no_grad():
    #             for batch in eval_loader:
    #                 x, y = batch
    #                 x, y = x.to(device), y.to(device)
    #                 logits = model(x).squeeze()
    #                 loss = criterion(logits, y)
    #                 eval_loss.append(loss.item())
    #                 print(f"Eval Loss: {loss.item()}")
    #                 res = compute_metrics(logits, y)
    #                 metrics["accuracy"].append(res["accuracy"])
    #                 metrics["f1"].append(res["f1"])
    #                 metrics["precision"].append(res["precision"])
    #                 metrics["recall"].append(res["recall"])
    #                 metrics["roc_auc"].append(res["roc_auc"])

    #                 break

    # import matplotlib.pyplot as plt
    # import seaborn as sns

    # sns.set_theme(style="darkgrid")
    # plt.figure(figsize=(10, 6))
    # plt.plot(train_loss, label="Train Loss")
    # plt.plot(eval_loss, label="Eval Loss")
    # plt.xlabel("Epoch")
    # plt.ylabel("Loss")
    # plt.title("Loss vs Epoch")
    # plt.legend()
    # plt.show()

    # plt.figure(figsize=(10, 6))
    # plt.plot(metrics["accuracy"], label="Accuracy")
    # plt.plot(metrics["f1"], label="F1")
    # plt.plot(metrics["precision"], label="Precision")
    # plt.plot(metrics["recall"], label="Recall")
    # plt.plot(metrics["roc_auc"], label="ROC AUC")
    # plt.xlabel("Epoch")
    # plt.ylabel("Metrics")
    # plt.title("Metrics vs Epoch")
    # plt.legend()
    # plt.show()


main()

C:\Users\shaya\AppData\Local\Temp\ipykernel_8316\51587222.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  classification_data = classification_data.replace("b", 0)
2025-04-09 12:47:46.128 | INFO     | src.models.protonet:fit:224 - Epoch 1/1000


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

ValueError: Target size (torch.Size([272])) must be the same as input size (torch.Size([272, 2]))